<img src="img/pandora2d_logo.png" width="500" height="500">

# Pandora2D : a coregistration framework

# Usage of deformation grid mode

#### Imports and external functions

In [ ]:
from pathlib import Path
from pprint import pprint
import numpy as np
from snippets.utils import *

#### Imports of pandora2d

In [ ]:
# Load pandora2d imports
from pandora2d import run 
from pandora2d.state_machine import Pandora2DMachine
from pandora2d.check_configuration import check_conf
from pandora2d.img_tools import create_datasets_from_inputs
from pandora2d.common import convert_disp_to_grid, convert_grid_to_disp

#### Load input data 

Provide image path

In [ ]:
# Paths to left and right images
img_left_path = "data/left.tif"
img_right_path = "data/right.tif"

Provide output directory to write results

In [ ]:
output_dir = Path.cwd() / "output"
# If necessary, create output dir
output_dir.mkdir(exist_ok=True, parents=True)

### Computation of deformation grids

Pandora2d allows you to activate a **deformation grid** mode in order to return deformation grids rather than disparity maps as output. 

To do this, you must enter the ``deformation_grid`` key in the Pandora2d output configuration. 

#### Instantiate the machine

In [ ]:
pandora2d_machine = Pandora2DMachine()

#### Define pipeline configuration with deformation grid mode

In [ ]:
user_cfg = {
    "input": {
        "left": {
            "img": img_left_path,
            "nodata": "NaN",
        },
        "right": {
            "img": img_right_path,
            "nodata": "NaN",
        },
        "col_disparity": {"init": 0, "range": 3},
        "row_disparity": {"init": 0, "range": 3},
    },
    "pipeline":{
        "matching_cost" : {
            "matching_cost_method": "zncc",
            "window_size": 7,
            "subpix":2
        },
        "disparity": {
            "disparity_method": "wta",
            "invalid_disparity": np.nan
        }
    },
    "output": {
        "path": "outputs/deformation_grid_mode",
        "deformation_grid": {"init_pixel_conv_grid": [0,0]}
    },
}

#### Check the user configuration

In [ ]:
checked_cfg = check_conf(user_cfg, pandora2d_machine)
pprint(checked_cfg)

#### Create image datasets

In [ ]:
image_datasets = create_datasets_from_inputs(input_config=checked_cfg["input"])

#### Visualize input data

In [ ]:
fig = plt.figure(figsize=(10,10))
ax0 = fig.add_subplot(1,2,1)
ax0.imshow(image_datasets.left["im"].data)
plt.title("Left image")
ax1 = fig.add_subplot(1,2,2)
ax1.imshow(image_datasets.right["im"].data)
plt.title("Right image")

#### Prepare the machine 

In [ ]:
pandora2d_machine.run_prepare(image_datasets.left, image_datasets.right, checked_cfg)

#### Trigger all the steps of the machine at ones

In [ ]:
dataset, output_cfg = run(pandora2d_machine, image_datasets.left, image_datasets.right, checked_cfg)

#### Convert disparity maps into deformation grids

When pandora2d is executed from the command line with the **deformation_grid** option, disparity maps are automatically converted into deformation grids.

Here, in API mode, to perform the conversion, you must call the ``convert_disp_to_grid`` method. 

In [ ]:
dataset = convert_disp_to_grid(dataset, output_cfg["output"]["deformation_grid"]["init_pixel_conv_grid"])

#### Visualize output deformation maps

In [ ]:
plot_two_images(dataset["row_deformation_map"].data,
                dataset["col_deformation_map"].data,
                "Row deformation map",
                "Column deformation map", 
                output_dir, 
                cmap=pandora_cmap_deformation_grid())

#### Convert deformation grids into disparity maps

It is also possible to convert deformation grids into disparity maps in API mode. 

To do this, simply use the ``convert_grid_to_disp`` method. 

In [ ]:
dataset = convert_grid_to_disp(dataset, output_cfg["output"]["deformation_grid"]["init_pixel_conv_grid"])

#### Visualize output disparity maps

In [ ]:
plot_two_images(dataset["row_map"].data,
                dataset["col_map"].data,
                "Row disparity map",
                "Columns disparity map", 
                output_dir, 
                cmap=pandora_cmap())